# Base de seis tablas
Genera `archivos/plantilla_reportes_base.xlsx` directamente desde las cuatro vistas SQL y Peopleware. No requiere otros notebooks ni Excel intermedios.

Las fechas se actualizan al ejecutar, usando Santiago:

- **Colas diario y Ejecutivos diario:** histórico disponible hasta hoy, con un límite de cinco años.
- **Colas y Estados por intervalo:** siete semanas completas hasta el último domingo, más la semana actual hasta hoy. La actual es la octava y se va completando.
- **Workloads diario:** histórico disponible desde el inicio de Peopleware (29/09/2025), limitado a cinco años, más el horizonte de forecast.
- **Workloads por intervalo:** siete semanas cerradas y seis semanas desde el lunes actual, para comparar y proyectar.

Los libros que guardes al cerrar un mes conservan el corte de esa ejecución. Guarda esos cierres como libros separados. La API aporta los datos que conserve; el rango solicitado no garantiza cobertura completa ni versiones anteriores del forecast.

Después puedes personalizar copias de la base. Conserva los nombres de las seis tablas y sus columnas de origen. Las duraciones son horas decimales, el TMO son segundos, y los nulos quedan vacíos.


In [1]:
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
from pathlib import Path
import polars as pl
from IPython.display import display
from analytics.flujo_reportes import cargar_datos, tablas_reportes

actual = Path.cwd().resolve()
BASE_DIR = next((p for p in (actual, *actual.parents)
                 if (p / "src" / "analytics").is_dir() and (p / "pyproject.toml").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Abrir el notebook dentro del proyecto.")
from analytics.flujo_reportes import crear_plantilla

HOY = datetime.now(ZoneInfo("America/Santiago")).date()
LUNES_ACTUAL = HOY - timedelta(days=HOY.weekday())
MAX_ANOS_HISTORIA = 5
SEMANAS_CERRADAS = 7
SEMANAS_WORKLOADS_DESDE_LUNES = 6
INICIO_PEOPLEWARE = date(2025, 9, 29)

try:
    DESDE = HOY.replace(year=HOY.year - MAX_ANOS_HISTORIA)
except ValueError:  # Aniversario del 29 de febrero en un año no bisiesto.
    DESDE = HOY.replace(year=HOY.year - MAX_ANOS_HISTORIA, day=28)
HASTA = HOY
DESDE_INTERVALOS = LUNES_ACTUAL - timedelta(weeks=SEMANAS_CERRADAS)
DESDE_WORKLOADS = max(DESDE, INICIO_PEOPLEWARE)
HASTA_WORKLOADS = LUNES_ACTUAL + timedelta(weeks=SEMANAS_WORKLOADS_DESDE_LUNES) - timedelta(days=1)
SEMANAS_INTERVALOS = SEMANAS_CERRADAS + 1

print(f"Diarios SQL: {DESDE} a {HASTA}, sólo registros disponibles")
print(f"Intervalos reales: {DESDE_INTERVALOS} a {HASTA}")
print(f"Workloads diario: {DESDE_WORKLOADS} a {HASTA_WORKLOADS}")
print(f"Workloads por intervalo: {max(DESDE_INTERVALOS, DESDE_WORKLOADS)} a {HASTA_WORKLOADS}")
RUTA_BASE = BASE_DIR / "archivos" / "plantilla_reportes_base.xlsx"


Diarios SQL: 2021-10-11 a 2026-10-11, sólo registros disponibles
Intervalos reales: 2026-08-17 a 2026-10-11
Workloads diario: 2025-09-29 a 2026-11-15
Workloads por intervalo: 2026-08-17 a 2026-11-15


In [2]:
resultado = cargar_datos(DESDE, HASTA, semanas_intervalos=SEMANAS_INTERVALOS,
                         desde_intervalos=DESDE_INTERVALOS,
                         desde_workloads=DESDE_WORKLOADS, hasta_workloads=HASTA_WORKLOADS,
                         base_dir=BASE_DIR)
display(pl.DataFrame({"fuente": list(resultado.datos),
                      "filas": [df.height for df in resultado.datos.values()],
                      "columnas": [df.width for df in resultado.datos.values()]}))
display(resultado.workloads.alertas)
display(resultado.workloads.cobertura_diaria)


Diarios: 2021-10-11 a 2026-10-11; intervalos SQL: 2026-08-17 a 2026-10-11.
Consultando las cuatro vistas SQL y el mapa de campañas...
Consultando Peopleware: 2025-09-29 a 2026-11-15...
colas_diario: 317,602 filas; 60 columnas.
colas: 92,582 filas; 40 columnas.
workloads_diario: 12,390 filas; 32 columnas.
workloads: 130,980 filas; 34 columnas.
ejecutivos: 438,637 filas; 37 columnas.
estados_agentes: 349,807 filas; 21 columnas.
Alertas de Workloads: 974.


fuente,filas,columnas
str,i64,i64
"""colas_diario""",317602,60
"""colas""",92582,40
"""workloads_diario""",12390,32
"""workloads""",130980,34
"""ejecutivos""",438637,37
"""estados_agentes""",349807,21


workload_id,motivo,detalle
str,str,str
"""834a9168-2ba4-4e2a-9ef0-f9c067…","""sin_unidad_asociada""","""Ceac grupo 1"""
"""267cbabf-fd23-433f-ac78-9cb58c…","""sin_unidad_asociada""","""Prueba OUT"""
"""4c4db074-dc55-43d3-b91f-878ce2…","""sin_curva_historica_identifica…","""Caec Outbound Hábil"""
"""0a68d7ea-2505-43c4-891e-e12c4f…","""sin_curva_historica_identifica…","""Ceac Grupo 2"""
"""4f8d1b69-1044-481b-9157-b9bbed…","""sin_curva_historica_identifica…","""Cruz Verde"""
…,…,…
"""eb8f36a4-1364-4409-be40-ff0c37…","""cobertura_diaria_tmo_faltante_…","""2026-10-27; tmo_adjustments"""
"""eb8f36a4-1364-4409-be40-ff0c37…","""cobertura_diaria_tmo_faltante_…","""2026-10-28; tmo_adjustments"""
"""eb8f36a4-1364-4409-be40-ff0c37…","""cobertura_diaria_tmo_faltante_…","""2026-10-29; tmo_adjustments"""


workload_id,workload,fecha,serie,valores,ceros,vacios,esperados,estado,peso,pesos_positivos,tmo_faltante_peso_positivo,horas_cubiertas,horas_dia
str,str,date,str,i64,i64,i64,i64,str,f64,i64,i64,f64,f64
"""d68d6e9e-439a-42b4-ba84-38209c…","""Bupa Seguros""",2025-09-29,"""llamadas_historicas""",96,60,0,96,"""completo""",null,null,null,null,null
"""d68d6e9e-439a-42b4-ba84-38209c…","""Bupa Seguros""",2025-09-29,"""contestadas_historicas""",96,60,0,96,"""completo""",null,null,null,null,null
"""d68d6e9e-439a-42b4-ba84-38209c…","""Bupa Seguros""",2025-09-29,"""tmo_historico""",96,60,0,96,"""completo""",557.0,36,0,null,null
"""d68d6e9e-439a-42b4-ba84-38209c…","""Bupa Seguros""",2025-09-29,"""forecast_raw_auto""",0,0,96,96,"""sin_datos""",null,null,null,null,null
"""d68d6e9e-439a-42b4-ba84-38209c…","""Bupa Seguros""",2025-09-29,"""tmo_raw_auto""",0,0,96,96,"""sin_pesos""",0.0,0,0,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a5832b0b-ce30-439b-93ab-39a1a0…","""WSA""",2026-11-15,"""forecast_adjustments""",0,0,96,96,"""sin_datos""",null,null,null,null,null
"""a5832b0b-ce30-439b-93ab-39a1a0…","""WSA""",2026-11-15,"""tmo_adjustments""",0,0,96,96,"""sin_pesos""",0.0,0,0,null,null
"""a5832b0b-ce30-439b-93ab-39a1a0…","""WSA""",2026-11-15,"""forecast_operational""",0,0,96,96,"""sin_datos""",null,null,null,null,null


In [3]:
# Guarda y cierra el libro de destino en Excel antes de regenerarlo.
crear_plantilla(resultado, RUTA_BASE)
print(f"Base creada y verificada: {RUTA_BASE}; seis hojas y seis tablas.")


Base creada y verificada: C:\Users\jorge.tapia\OneDrive - Sercomed Servicios virtuales de Salud\Documentos\desarrollos_python\librerías\xlwings\reportes_excel_desde_plantilla\archivos\plantilla_reportes_base.xlsx; seis hojas y seis tablas.
